In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans


In [3]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

!mkdir Celegans_chemical_gene
# ── Output path ───────────────────────────────────────────────────────────────

OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans/Celegans_chemical_gene/Celegans_chemical_gene.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

mkdir: cannot create directory ‘Celegans_chemical_gene’: File exists


In [4]:
#

In [5]:
# Reference database paths
PUBCHEM_SYN_PATH  = f"{BASE_DIR}data_collection/databases_for_mapping/pubchem/CID-Synonym-filtered"
PUBCHEM_PKL_PATH  = f"{BASE_DIR}data_collection/databases_for_mapping/pubchem/combined_df.pkl"

# Used to resolve compound names to PubChem CIDs
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))  # synonym → CID
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}  # lowercase version for case-insensitive lookup

# ── PubChem CID → IUPAC name and SMILES ──────────────────────────
Pubchem               = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))

In [ ]:
Pubchem_IUPAC_CID_Dict

# stitch

In [ ]:
stitch = pd.read_csv(PROC_DIR + 'stitch/stitch_CELE_CHEMICALENTITY_GENE.csv')
stitch.columns = stitch.columns.str.lower()

stitch['head_detail_name'] = (
    stitch['head_detail_name']
    .fillna(stitch['head'].astype(str).map(Pubchem_IUPAC_CID_Dict))
)
stitch['kg_type'] = 'Generalised'
stitch['species'] = 'C.elegans'

print(f"stitch: {len(stitch):,} rows")
stitch

In [8]:
stitch[stitch['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_type


# Consolidate into Unified Schem

In [ ]:
# List all source DataFrames to include
source_dfs = [
    stitch
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

# Sanity Check — Distinct Values

In [ ]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

In [11]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 9,845,825 remaining


# NaN Audit (pre-dedup)

In [12]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,9845825,9845825,19691650
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [13]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 9,845,825  |  After dedup: 9,845,825


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_Gene,B0024.12,ChemicalEntity,NaN,Gene,STITCH,Generalised,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Glucosamine 6-phosphate N-acetyltransferase,C.elegans
1,1,ChemicalEntity_Gene,B0041.7,ChemicalEntity,NaN,Gene,STITCH,Generalised,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Transcriptional regulator ATRX homolog,C.elegans
2,1,ChemicalEntity_Gene,B0272.4,ChemicalEntity,NaN,Gene,STITCH,Generalised,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Uncharacterized protein,C.elegans


In [14]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,9845825,9845825,19691650
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [15]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'STITCH'} kg_source
STITCH    9845825
Name: count, dtype: int64


In [16]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    9845825
Name: count, dtype: int64


In [17]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 9,845,825 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans/Celegans_chemical_gene/Celegans_chemical_gene.csv


In [18]:
#